# 🏦 Alfa Bank — Business Client Segment Classifier
### Ступень 1: Мультиклассовая классификация бизнес-портретов

**Ячейки:**
1. Установка зависимостей
2. Конфиг и константы
3. Загрузка данных
4. Обучение модели
5. Метрики и валидация
6. Инференс + Top-5 SHAP признаки

## 📦 1. Установка зависимостей

In [1]:
!pip install catboost scikit-learn pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


## ⚙️ 2. Конфиг и константы

In [2]:
import warnings
import pickle
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# ── Пути к файлам ────────────────────────────────────────────────
DATA_PATH  = 'alfa_onboarding_dataset_5000.csv'  # загрузите через Files → Upload
MODEL_PATH = 'alfa_classifier.cbm'
META_PATH  = 'alfa_classifier_meta.pkl'

TARGET_COL = 'target'

# ── Категориальные фичи (CatBoost обрабатывает нативно) ──────────
CAT_FEATURES = [
    'smb_type_code',
    'main_okved',
    'okved_major',
    'okved_major_wrapped',
    'categ_name',
    'srvpackage_sale_uk',
    'sourceattr_ccode',
    'city',
    'addrf_region_name',
    'division_name',
    'registration_reason_code',
]

# ── Описания фичей (для SHAP-вывода) ─────────────────────────────
FEATURE_DESCRIPTIONS = {
    'smb_type_code':                   'Тип субъекта МСП (ЮЛ / ИП / КФХ)',
    'main_okved':                      'Основной ОКВЭД',
    'okved_major':                     'Отраслевая группа ОКВЭД (2 знака)',
    'okved_major_wrapped':             'Отрасль основного ОКВЭД',
    'okved_cnt_total':                 'Количество всех ОКВЭД',
    'okved_groups_unique_cnt':         'Количество уникальных групп ОКВЭД',
    'okved_groups_share':              'Доля основных групп ОКВЭД',
    'days_from_ogrn':                  'Дней с регистрации бизнеса (ОГРН)',
    'days_from_smb':                   'Дней в реестре МСП',
    'ogrn_days_end_month':             'Дней с регистрации ОГРН до конца месяца',
    'ogrn_days_end_quarter':           'Дней с регистрации ОГРН до конца квартала',
    'week_sum_transactions':           'Сумма дебетовых транзакций за неделю (₽)',
    'week_mean_transactions':          'Количество транзакций за неделю',
    'share_last_month':                'Доля транзакций за последний месяц',
    'share_last_3_months':             'Доля транзакций за 3 месяца',
    'categ_name':                      'Категория кэшбэка / расходов',
    'acquiring_num_live':              'Действующих эквайринговых продуктов',
    'zpp_num_live':                    'Действующих зарплатных проектов',
    'nkop_num_live':                   'Действующих продуктов «Налоговая копилка»',
    'rko_num_live':                    'Действующих расчётных счетов',
    'srvpackage_sale_uk':              'Пакет услуг на старте',
    'sourceattr_ccode':                'Канал привлечения клиента',
    'city':                            'Город',
    'addrf_region_name':               'Регион',
    'postindex_town_code':             'Почтовый код города',
    'postindex_area_code':             'Почтовый код района',
    'oktmo_reg_code':                  'Код региона ОКТМО',
    'oktmo_ccode':                     'Код ОКТМО',
    'oktmo_municipality_code':         'Код муниципалитета ОКТМО',
    'kpp_region_code':                 'Регион постановки на учёт (КПП)',
    'addrf_region_code':               'Код региона проживания',
    'branch_eq_ccode':                 'Код филиала банка',
    'sks_city':                        'Код города (СКС)',
    'division_name':                   'Клиентское подразделение',
    'regorg_code':                     'Код органа регистрации (СОУН)',
    'registration_reason_code':        'Причина постановки на учёт (КПП)',
    'pensfund_authority_code':         'Код терр. органа Пенсионного фонда',
    'apin_salary_last_days':           'Среднее дней с последнего зачисления ФОТ',
    'apin_product_active_days':        'Среднее дней с даты активации продукта',
    'xpin_start_days':                 'Среднее дней с даты открытия руководителей',
    'xpin_birth_days':                 'Среднее дней с даты рождения руководителей',
    'days_from_authperson_registration': 'Дней регистрации управляющего',
    'apin_status_period_cnt':          'Среднее месяцев в статусе у руководителей',
    'prev_managers':                   'Количество предыдущих управляющих',
    'sparkcompany_uk':                 'ID клиента в системе СПАРК',
    'accum':                           'Признак накопительного продукта',
    'impnt':                           'Индекс важности клиента',
    'to_activate':                     'Флаг требует активации',
    'complexity':                      'Сложность профиля клиента',
    'get_scores':                      'Скоринговый балл',
}

# ── Описания сегментов ────────────────────────────────────────────
CLASS_DESCRIPTIONS = {
    'P1': '🏪 Розничный продавец',
    'P2': '💻 IT / Онлайн-бизнес',
    'P3': '🏗️  Строитель / Подрядчик',
    'P4': '🎓 Фрилансер / Самозанятый ИП',
    'P5': '🌱 Новое ЮЛ / Стартап',
    'P6': '🍽️  Услуги / HoReCa',
    'P7': '🚚 Логистика / Транспорт',
    'P8': '🌾 Агробизнес / КФХ',
}

print('✅ Конфиг загружен')

✅ Конфиг загружен


## 📂 3. Загрузка и подготовка данных

In [3]:
def load_data(path):
    df = pd.read_csv(path, dtype=str)
    for col in df.columns:
        if col not in CAT_FEATURES and col != TARGET_COL:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in CAT_FEATURES:
        if col in df.columns:
            df[col] = df[col].astype(str)
    return df


df = load_data(DATA_PATH)

feature_cols = [c for c in df.columns if c != TARGET_COL]
cat_indices  = [i for i, c in enumerate(feature_cols) if c in CAT_FEATURES]

X = df[feature_cols]
y = df[TARGET_COL]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Датасет: {df.shape[0]} строк × {df.shape[1]} колонок')
print(f'Train: {len(X_train)}  |  Val: {len(X_val)}')
print(f'\nРаспределение классов:')
print(df[TARGET_COL].value_counts().sort_index())

Датасет: 5000 строк × 51 колонок
Train: 4000  |  Val: 1000

Распределение классов:
target
P1    625
P2    625
P3    625
P4    625
P5    625
P6    625
P7    625
P8    625
Name: count, dtype: int64


## 🤖 4. Обучение CatBoost-классификатора

In [4]:
train_pool = Pool(X_train, y_train, cat_features=cat_indices)
val_pool   = Pool(X_val,   y_val,   cat_features=cat_indices)

model = CatBoostClassifier(
    iterations            = 800,
    learning_rate         = 0.05,
    depth                 = 7,
    l2_leaf_reg           = 3.0,
    loss_function         = 'MultiClass',
    eval_metric           = 'Accuracy',
    random_seed           = 42,
    early_stopping_rounds = 80,
    auto_class_weights    = 'Balanced',
    task_type             = 'CPU',
    verbose               = 100,
)

model.fit(
    train_pool,
    eval_set       = val_pool,
    use_best_model = True,
)

# ── Сохранение модели ─────────────────────────────────────────────
model.save_model(MODEL_PATH)

meta = {
    'feature_cols': feature_cols,
    'cat_indices':  cat_indices,
    'classes':      list(model.classes_),
}
with open(META_PATH, 'wb') as f:
    pickle.dump(meta, f)

print(f'\n✅ Модель сохранена → {MODEL_PATH}')
print(f'✅ Мета-данные сохранены → {META_PATH}')

0:	learn: 0.9922500	test: 0.9970000	best: 0.9970000 (0)	total: 511ms	remaining: 6m 48s
100:	learn: 0.9997500	test: 1.0000000	best: 1.0000000 (83)	total: 40.1s	remaining: 4m 37s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 1
bestIteration = 83

Shrink model to first 84 iterations.

✅ Модель сохранена → alfa_classifier.cbm
✅ Мета-данные сохранены → alfa_classifier_meta.pkl


## 📊 5. Метрики и валидация

In [5]:
y_pred = model.predict(X_val).flatten()

acc = accuracy_score(y_val, y_pred)
f1  = f1_score(y_val, y_pred, average='macro')

print(f'Accuracy : {acc:.4f}')
print(f'F1-macro : {f1:.4f}')
print()
print(classification_report(y_val, y_pred))

Accuracy : 1.0000
F1-macro : 1.0000

              precision    recall  f1-score   support

          P1       1.00      1.00      1.00       125
          P2       1.00      1.00      1.00       125
          P3       1.00      1.00      1.00       125
          P4       1.00      1.00      1.00       125
          P5       1.00      1.00      1.00       125
          P6       1.00      1.00      1.00       125
          P7       1.00      1.00      1.00       125
          P8       1.00      1.00      1.00       125

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [6]:
# ── Глобальная важность признаков ────────────────────────────────
importances = model.get_feature_importance()
feat_imp = sorted(zip(feature_cols, importances), key=lambda x: -x[1])

print('TOP-15 GLOBAL FEATURE IMPORTANCE\n')
for rank, (feat, imp) in enumerate(feat_imp[:15], 1):
    desc = FEATURE_DESCRIPTIONS.get(feat, feat)
    bar  = '█' * int(imp / 2)
    print(f'  {rank:2d}. {feat:<40s} {imp:5.2f}%  {bar}')
    print(f'       {desc}')

TOP-15 GLOBAL FEATURE IMPORTANCE

   1. okved_major_wrapped                      42.09%  █████████████████████
       Отрасль основного ОКВЭД
   2. okved_major                              36.27%  ██████████████████
       Отраслевая группа ОКВЭД (2 знака)
   3. sourceattr_ccode                         13.01%  ██████
       Канал привлечения клиента
   4. days_from_ogrn                            2.83%  █
       Дней с регистрации бизнеса (ОГРН)
   5. apin_salary_last_days                     2.59%  █
       Среднее дней с последнего зачисления ФОТ
   6. main_okved                                1.36%  
       Основной ОКВЭД
   7. smb_type_code                             0.41%  
       Тип субъекта МСП (ЮЛ / ИП / КФХ)
   8. week_sum_transactions                     0.37%  
       Сумма дебетовых транзакций за неделю (₽)
   9. addrf_region_name                         0.23%  
       Регион
  10. city                                      0.19%  
       Город
  11. ogrn_days_end_month   

## 🔍 6. Инференс + Top-5 SHAP для конкретного клиента

In [7]:
def prepare_sample(raw, feature_cols):
    row = {}
    for col in feature_cols:
        val = raw.get(col, np.nan)
        if col in CAT_FEATURES:
            row[col] = str(val) if val is not None else 'nan'
        else:
            try:
                row[col] = float(val)
            except (TypeError, ValueError):
                row[col] = np.nan
    return pd.DataFrame([row], columns=feature_cols)


def get_top5_shap(model, sample_df, cat_indices, predicted_class, classes):
    pool = Pool(sample_df, cat_features=cat_indices)

    # shape: (n_samples, n_features + 1, n_classes)  — последний столбец bias
    shap_vals   = model.get_feature_importance(data=pool, type='ShapValues', shap_calc_type='Regular')
    class_idx   = list(classes).index(predicted_class)
    shap_sample = shap_vals[0, :-1, class_idx]

    ranked = sorted(enumerate(shap_sample), key=lambda x: abs(x[1]), reverse=True)[:5]

    result = []
    for rank, (feat_idx, shap_val) in enumerate(ranked, 1):
        feat_name = feature_cols[feat_idx]
        result.append({
            'rank':        rank,
            'feature':     feat_name,
            'value':       sample_df.iloc[0][feat_name],
            'shap':        round(float(shap_val), 5),
            'direction':   '▲ в пользу сегмента' if shap_val > 0 else '▼ против сегмента',
            'description': FEATURE_DESCRIPTIONS.get(feat_name, feat_name),
        })
    return result


def predict(raw_sample):
    sample_df = prepare_sample(raw_sample, feature_cols)
    pool      = Pool(sample_df, cat_features=cat_indices)

    proba    = model.predict_proba(pool)[0]
    pred_idx = int(np.argmax(proba))
    pred_cls = model.classes_[pred_idx]

    top5 = get_top5_shap(model, sample_df, cat_indices, pred_cls, model.classes_)

    # ── Вывод ────────────────────────────────────────────────────
    print('=' * 60)
    print(f"  Сегмент   : {pred_cls}  {CLASS_DESCRIPTIONS.get(pred_cls, '')}")
    print(f'  Уверенность: {proba[pred_idx]*100:.1f}%')

    print('\n  Вероятности по всем сегментам:')
    for cls, p in sorted(zip(model.classes_, proba), key=lambda x: -x[1]):
        bar = '█' * int(p * 30)
        print(f'    {cls}  {CLASS_DESCRIPTIONS.get(cls, ""):<32s}  {p*100:5.1f}%  {bar}')

    print('\n' + '─' * 60)
    print('  TOP-5 SHAP — ключевые признаки решения')
    print('─' * 60)
    for item in top5:
        print(f"  #{item['rank']}  {item['feature']:<42s}  val={str(item['value']):<18s}  shap={item['shap']:+.4f}  {item['direction']}")
        print(f"       └─ {item['description']}")

    return {
        'predicted_class':         pred_cls,
        'class_description':       CLASS_DESCRIPTIONS.get(pred_cls, ''),
        'confidence':              round(float(proba[pred_idx]), 4),
        'probabilities':           {cls: round(float(p), 4) for cls, p in zip(model.classes_, proba)},
        'top5_feature_importance': top5,
    }


print('✅ Функции инференса готовы')

✅ Функции инференса готовы


In [8]:
# ── Пример: Розничный ИП ─────────────────────────────────────────
# Замените значения на данные реального клиента

sample = {
    'smb_type_code':                   '2',           # ИП
    'main_okved':                      '47',          # Розничная торговля
    'okved_major':                     '47',
    'okved_major_wrapped':             'retail_trade',
    'okved_cnt_total':                 2,
    'okved_groups_unique_cnt':         2,
    'okved_groups_share':              0.8,
    'days_from_ogrn':                  400,
    'days_from_smb':                   380,
    'ogrn_days_end_month':             12,
    'ogrn_days_end_quarter':           40,
    'week_sum_transactions':           130000,
    'week_mean_transactions':          18,
    'share_last_month':                0.55,
    'share_last_3_months':             0.72,
    'categ_name':                      'grocery',
    'acquiring_num_live':              '0',
    'zpp_num_live':                    '0',
    'nkop_num_live':                   '0',
    'rko_num_live':                    '1',
    'srvpackage_sale_uk':              'standard',
    'sourceattr_ccode':                'branch',
    'city':                            'Краснодар',
    'addrf_region_name':               'Краснодарский край',
    'postindex_town_code':             350000,
    'postindex_area_code':             350,
    'oktmo_reg_code':                  23,
    'oktmo_ccode':                     37000000,
    'oktmo_municipality_code':         3712,
    'kpp_region_code':                 23,
    'addrf_region_code':               23,
    'branch_eq_ccode':                 45,
    'sks_city':                        120,
    'division_name':                   'МСБ_Юг',
    'regorg_code':                     2310,
    'registration_reason_code':        '1',
    'pensfund_authority_code':         23,
    'apin_salary_last_days':           120,
    'apin_product_active_days':        350,
    'xpin_start_days':                 900,
    'xpin_birth_days':                 14600,
    'days_from_authperson_registration': 700,
    'apin_status_period_cnt':          10,
    'prev_managers':                   1,
    'sparkcompany_uk':                 4123456,
    'accum':                           0.3,
    'impnt':                           0.6,
    'to_activate':                     1,
    'complexity':                      0.4,
    'get_scores':                      0.62,
}

result = predict(sample)

  Сегмент   : P1  🏪 Розничный продавец
  Уверенность: 98.1%

  Вероятности по всем сегментам:
    P1  🏪 Розничный продавец               98.1%  █████████████████████████████
    P2  💻 IT / Онлайн-бизнес                0.3%  
    P3  🏗️  Строитель / Подрядчик           0.3%  
    P8  🌾 Агробизнес / КФХ                  0.3%  
    P5  🌱 Новое ЮЛ / Стартап                0.3%  
    P6  🍽️  Услуги / HoReCa                 0.3%  
    P7  🚚 Логистика / Транспорт             0.3%  
    P4  🎓 Фрилансер / Самозанятый ИП        0.3%  

────────────────────────────────────────────────────────────
  TOP-5 SHAP — ключевые признаки решения
────────────────────────────────────────────────────────────
  #1  smb_type_code                               val=2                   shap=+0.0578  ▲ в пользу сегмента
       └─ Тип субъекта МСП (ЮЛ / ИП / КФХ)
  #2  okved_cnt_total                             val=2.0                 shap=-0.0317  ▼ против сегмента
       └─ Количество всех ОКВЭД
  #3  okved_majo